# 专家方差形状分析

读取 `expert_variance_probe.jsonl`，对每条序列（每个 prompt）计算每个 token 位置上的两类方差并画图：

- **专家间方差 (between-expert)**：4 个专家（chasing / long_term / repurchase / generalized）在每个候选 ID 上的方差，再对 20 个候选 ID 取均值。反映"专家之间在哪里分歧"。
- **专家内方差 (within-expert)**：每个专家自身 20 个候选 ID 概率分布的方差，再对 4 个专家取均值。反映"每个专家的分布有多尖锐"。

预处理：先对每个专家每个 token 的 top-20 概率做 **top-k=20 归一化**（`p / sum(p)`），使 20 个值和为 1，隔离"形状"与原始尺度。`expert_base` / `student_base` 不参与方差（前者是无专家指令的基线，后者是学生），仅作参考。

> 方差用样本方差（ddof=1，除以自由度）：专家间 df=3，专家内 df=19。

In [ ]:
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# CJK 字体（Mac 本地绘图用；服务器若无 CJK 字体可忽略警告，英文部分仍可读）
if platform.system() == "Darwin":
    matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti TC", "Arial Unicode MS", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

# 找 JSONL：优先 ~/Downloads（本地），其次 scripts/（服务器同目录），再 cwd
_candidates = [
    Path.home() / "Downloads" / "expert_variance_probe.jsonl",
    Path("scripts") / "expert_variance_probe.jsonl",
    Path("expert_variance_probe.jsonl"),
]
JSONL = next((p for p in _candidates if p.exists()), None)
assert JSONL is not None, f"未找到 JSONL，请把 expert_variance_probe.jsonl 放到: {_candidates}"
print("loading:", JSONL)

_rows = []
with open(JSONL, encoding="utf-8") as _f:
    for _line in _f:
        _rows.append(json.loads(_line))
df = pd.DataFrame(_rows)
print("rows:", len(df))
print("sequences (user_id x sample_idx):", df.groupby(['user_id', 'sample_idx']).ngroups)
print("label dist:", df['label'].value_counts().to_dict())
print("field dist:", df['field'].value_counts().to_dict())
df.head()

## 计算每个 token 的两类方差

对每条序列独立计算（不跨序列混合），输出 `seq_var`：每个 token 一行，含 `between`、`within`、`label`、`field`、`token_text`。

In [ ]:
EXPERTS = ["expert_chasing", "expert_long_term", "expert_repurchase", "expert_generalized"]

_seq_frames = []
for (uid, sidx), g in df.groupby(["user_id", "sample_idx"]):
    g = g.sort_values("token_pos").reset_index(drop=True)
    # P: (T, 4, 20) — 4 个专家 × 20 个候选 ID 的原始概率
    P = np.stack([np.stack(g[e].tolist()) for e in EXPERTS], axis=1)
    # top-k=20 归一化：每个专家每个 token 的 20 个概率归一化到和为 1
    P = P / P.sum(axis=2, keepdims=True)
    # 专家间：每 dim 上 4 个专家的样本方差（df=3），再对 20 dim 取均值
    mean_across = P.mean(axis=1)                                   # (T, 20)
    between_per_dim = ((P - mean_across[:, None, :]) ** 2).sum(axis=1) / (4 - 1)  # (T, 20)
    between = between_per_dim.mean(axis=1)                         # (T,)
    # 专家内：每个专家 20 个值的样本方差（df=19），再对 4 个专家取均值
    within_per_expert = P.var(axis=2, ddof=1)                      # (T, 4)
    within = within_per_expert.mean(axis=1)                        # (T,)
    _seq_frames.append(pd.DataFrame({
        "user_id": uid,
        "sample_idx": sidx,
        "stratum": g["stratum"].iloc[0],
        "token_pos": g["token_pos"].values,
        "token_text": g["token_text"].values,
        "label": g["label"].values,
        "field": g["field"].values,
        "between": between,
        "within": within,
    }))
seq_var = pd.concat(_seq_frames, ignore_index=True)
N_SEQS = df.groupby(["user_id", "sample_idx"]).ngroups
print("seq_var rows:", len(seq_var), "| sequences:", N_SEQS)
seq_var[["user_id", "token_pos", "token_text", "label", "field", "between", "within"]].head(8)

## 图 1：每条序列的方差形状

每个子图一条序列，x = token 位置。红 = 专家间方差（左 y 轴），蓝 = 专家内方差（右 y 轴，二者尺度不同故双轴）。▼ 标记类目 token（label=1）。

In [ ]:
_seq_keys = list(df.groupby(["user_id", "sample_idx"]).groups.keys())
_n = len(_seq_keys)
_ncols = 3
_nrows = (_n + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(6.4 * _ncols, 3.6 * _nrows))
axes = np.array(axes).reshape(-1)
for ax, (uid, sidx) in zip(axes, _seq_keys):
    g = seq_var[(seq_var.user_id == uid) & (seq_var.sample_idx == sidx)]
    x = g["token_pos"].to_numpy()
    ax.plot(x, g["between"].to_numpy(), color="#d62728", lw=1.0, label="between")
    ax2 = ax.twinx()
    ax2.plot(x, g["within"].to_numpy(), color="#1f77b4", lw=1.0, alpha=0.85, label="within")
    cat = g[g["label"] == 1]
    ax.scatter(cat["token_pos"], cat["between"], color="#d62728", s=16, marker="v", zorder=5)
    ax2.scatter(cat["token_pos"], cat["within"], color="#1f77b4", s=16, marker="v", zorder=5)
    ax.set_title(f"{uid}\n{g['stratum'].iloc[0]}  (len {len(g)})", fontsize=9)
    ax.set_xlabel("token pos")
    ax.set_ylabel("专家间 between", color="#d62728")
    ax2.set_ylabel("专家内 within", color="#1f77b4")
    ax.tick_params(axis="y", labelcolor="#d62728")
    ax2.tick_params(axis="y", labelcolor="#1f77b4")
for ax in axes[_n:]:
    ax.axis("off")
fig.suptitle("Per-sequence variance shape — between (red/left) vs within (blue/right); v = category token", fontsize=11)
fig.tight_layout()
plt.show()

## 图 2：跨序列平均方差形状

把 6 条序列按 token 位置对齐（截到最短序列长度），画均值曲线 + std 带。看"平均而言"方差沿位置的形态。

In [ ]:
_L = int(seq_var.groupby(["user_id", "sample_idx"]).size().min())
_trunc = seq_var[seq_var["token_pos"] < _L]
_agg = _trunc.groupby("token_pos")[["between", "within"]].agg(["mean", "std"])
_pos = _agg.index.to_numpy()
_b_m = _agg[("between", "mean")].to_numpy(); _b_s = _agg[("between", "std")].to_numpy()
_w_m = _agg[("within", "mean")].to_numpy();  _w_s = _agg[("within", "std")].to_numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(_pos, _b_m, color="#d62728", label="专家间 between (mean)")
ax.fill_between(_pos, _b_m - _b_s, _b_m + _b_s, color="#d62728", alpha=0.15)
ax.plot(_pos, _w_m, color="#1f77b4", label="专家内 within (mean)")
ax.fill_between(_pos, _w_m - _w_s, _w_m + _w_s, color="#1f77b4", alpha=0.15)
ax.set_xlabel(f"token pos (truncated to min len = {_L})")
ax.set_ylabel("variance")
ax.set_title(f"Cross-sequence mean variance shape (n={N_SEQS} seqs, first {_L} tokens)")
ax.legend()
fig.tight_layout()
plt.show()

## 图 3：按 token 类型看方差分布

类目 token（label=1） vs 理由 token（field=理由） vs 结构 token（structure） 的方差箱线图。看专家分歧 / 尖锐度是否在类目 token 上更明显。

In [ ]:
_groups = {
    "类目 (label=1)": seq_var[seq_var["label"] == 1],
    "理由 (reason)": seq_var[seq_var["field"] == "理由"],
    "结构 (structure)": seq_var[seq_var["field"] == "structure"],
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, title in zip(axes, ["between", "within"], ["专家间方差 between", "专家内方差 within"]):
    data = [_g[col].to_numpy() for _g in _groups.values()]
    ax.boxplot(data, labels=list(_groups.keys()), showfliers=False)
    ax.set_title(title)
    ax.set_ylabel("variance")
    ax.tick_params(axis="x", labelrotation=10)
fig.suptitle("Variance by token type")
fig.tight_layout()
plt.show()

## 图 4：专家间 vs 专家内 散点

每个 token 一个点，颜色按 label，log-log。看两个信号是否相关：尖锐分布（高 within）是否伴随专家分歧（高 between）。

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
for lab, c, name in [(1, "#d62728", "类目 category"), (0, "#1f77b4", "非类目 non-category")]:
    sub = seq_var[seq_var["label"] == lab]
    ax.scatter(np.maximum(sub["between"].to_numpy(), 1e-12),
               np.maximum(sub["within"].to_numpy(), 1e-12),
               s=6, alpha=0.4, c=c, label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("专家间方差 between")
ax.set_ylabel("专家内方差 within")
ax.set_title("between vs within (log-log)")
ax.legend()
fig.tight_layout()
plt.show()

## 图 5：每个专家的 within 方差曲线

把 within 方差按专家拆开（不取 4 专家均值），看哪一位专家的分布在序列上最尖锐/最平坦。

In [ ]:
# 重新计算每个专家的 within（不平均），按序列画
fig, axes = plt.subplots(2, 3, figsize=(6.4 * 3, 3.4 * 2))
axes = np.array(axes).reshape(-1)
_colors = {"expert_chasing": "#d62728", "expert_long_term": "#ff7f0e",
           "expert_repurchase": "#2ca02c", "expert_generalized": "#9467bd"}
for ax, (uid, sidx) in zip(axes, _seq_keys):
    g = df[(df.user_id == uid) & (df.sample_idx == sidx)].sort_values("token_pos").reset_index(drop=True)
    x = g["token_pos"].to_numpy()
    for e in EXPERTS:
        P = np.stack(g[e].tolist())              # (T, 20)
        P = P / P.sum(axis=1, keepdims=True)
        w = P.var(axis=1, ddof=1)                 # (T,)
        ax.plot(x, w, color=_colors[e], lw=0.9, alpha=0.85, label=e.replace("expert_", ""))
    ax.set_title(f"{uid}  ({g['stratum'].iloc[0]})", fontsize=9)
    ax.set_xlabel("token pos"); ax.set_ylabel("within var")
    ax.legend(fontsize=7, loc="upper right")
fig.suptitle("Per-expert within-variance shape (top-20 normalized)", fontsize=11)
fig.tight_layout()
plt.show()

## 解读提示

- **专家间方差高**的位置 = 4 个专家在这里分歧大（prompt 差异显化）。
- **专家内方差高**的位置 = 每个专家的 top-20 分布尖锐（某个候选 ID 主导）。
- 类目 token（▼）上通常 within 高（teacher 把概率集中到某个类目候选），between 也常偏高（专家偏向不同类目）。
- 若某段连续 structure / 理由 token 上 between ≈ 0，说明 4 个专家在"非类目"部分几乎一致——分歧主要发生在类目选择上。
- 图 5 拆分专家后，若某专家曲线系统性偏高，说明该专家指令让 teacher 更笃定（分布更尖锐）。